# Project 2 — Advanced Solution: Modular Arithmetic

**Python:** modern Python 3.11+ • **Dependencies:** standard library only • **Run:** Kernel → Restart & Run All

## 1. Requirements, contract, and deliberate decisions

The supplied exercise builds a `Mod(value, modulus)` class, adds arithmetic and ordering, then refactors repetitive checks via a coercion helper and the `operator` module. This notebook gives one complete, self-contained implementation rather than retaining multiple overwritten class definitions.

**Preserved:** mutable `value`, read-only `modulus`, normalized residues, equality/congruence with integers, residue-based comparisons, `+`, `-`, `*`, `**`, and their in-place variants. **Added:** reflected operations, rejection of `bool`, efficient and mathematically correct exponentiation, negative exponents when inverses exist, safe failed in-place operations, and comprehensive tests.

**Important caveat:** modulo classes with different moduli are incompatible for arithmetic; they compare unequal for equality and raise `TypeError` for ordering. Integer equality is congruence (`Mod(1, 5) == 6`), not conventional integer identity. This mixed-modulus / integer relation is not a global equivalence relation; don't treat it as one. The mutable class is **unhashable** to avoid breaking dictionary/set invariants. The extended notebook demonstrates an immutable, coherently hashable alternative.

## 2. Implementation

A shared `_coerce` helper returns a compatible integer residue or `NotImplemented`, allowing Python to try reflected operators. `_operate` factors out arithmetic and in-place updates. Exponentiation intentionally has a dedicated implementation, using `pow(base, exponent, modulus)` instead of building enormous intermediate integers or incorrectly reducing exponents.

In [1]:
from __future__ import annotations

import operator
from functools import total_ordering
from types import NotImplementedType
from typing import Callable


@total_ordering
class Mod:
    """Mutable residue modulo a positive integer.

    Integer operands are reduced modulo ``self.modulus``. A Mod operand must
    have the same modulus. Equality with integers means congruence, matching
    the source exercise. Therefore this class is intentionally UNHASHABLE:
    mutable, and congruence with distinct integers violates the hash contract.
    """

    __slots__ = ("_value", "_modulus")
    __hash__ = None

    def __init__(self, value: int, modulus: int) -> None:
        self._check_int(modulus, "modulus")
        if modulus <= 0:
            raise ValueError("modulus must be strictly positive")
        self._check_int(value, "value")
        self._modulus = modulus
        self._value = value % modulus

    @staticmethod
    def _check_int(value: object, name: str) -> None:
        # bool subclasses int, but True is not an appropriate input here.
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError(f"{name} must be an integer (excluding bool)")

    @property
    def modulus(self) -> int:
        """Read-only modulus."""
        return self._modulus

    @property
    def value(self) -> int:
        """Canonical residue in range [0, modulus)."""
        return self._value

    @value.setter
    def value(self, new_value: int) -> None:
        self._check_int(new_value, "value")
        self._value = new_value % self.modulus

    def __repr__(self) -> str:
        return f"Mod({self.value}, {self.modulus})"

    def __int__(self) -> int:
        return self.value

    def _coerce(self, other: object) -> int | NotImplementedType:
        """Return a compatible residue, or defer to Python's dispatch."""
        if isinstance(other, Mod):
            return other.value if self.modulus == other.modulus else NotImplemented
        if isinstance(other, int) and not isinstance(other, bool):
            return other % self.modulus
        return NotImplemented

    def _operate(
        self,
        other: object,
        op: Callable[[int, int], int],
        *,
        reverse: bool = False,
        in_place: bool = False,
    ) -> Mod | NotImplementedType:
        operand = self._coerce(other)
        if operand is NotImplemented:
            return NotImplemented
        left, right = (operand, self.value) if reverse else (self.value, operand)
        result = op(left, right) % self.modulus
        if in_place:
            self._value = result
            return self
        return Mod(result, self.modulus)

    def __eq__(self, other: object) -> bool:
        if isinstance(other, Mod) and self.modulus != other.modulus:
            return False
        operand = self._coerce(other)
        return NotImplemented if operand is NotImplemented else self.value == operand

    def __lt__(self, other: object) -> bool:
        operand = self._coerce(other)
        return NotImplemented if operand is NotImplemented else self.value < operand

    def __neg__(self) -> Mod:
        return Mod(-self.value, self.modulus)

    def __pos__(self) -> Mod:
        return Mod(self.value, self.modulus)

    def __add__(self, other: object) -> Mod:
        return self._operate(other, operator.add)

    def __radd__(self, other: object) -> Mod:
        return self._operate(other, operator.add, reverse=True)

    def __iadd__(self, other: object) -> Mod:
        return self._operate(other, operator.add, in_place=True)

    def __sub__(self, other: object) -> Mod:
        return self._operate(other, operator.sub)

    def __rsub__(self, other: object) -> Mod:
        return self._operate(other, operator.sub, reverse=True)

    def __isub__(self, other: object) -> Mod:
        return self._operate(other, operator.sub, in_place=True)

    def __mul__(self, other: object) -> Mod:
        return self._operate(other, operator.mul)

    def __rmul__(self, other: object) -> Mod:
        return self._operate(other, operator.mul, reverse=True)

    def __imul__(self, other: object) -> Mod:
        return self._operate(other, operator.mul, in_place=True)

    def __pow__(self, exponent: object, modulo: object = None) -> Mod:
        if modulo is not None:
            raise TypeError("Mod already defines the modulus; use base ** exponent")
        exp = self._coerce(exponent) if isinstance(exponent, Mod) else exponent
        if exp is NotImplemented or not isinstance(exp, int) or isinstance(exp, bool):
            return NotImplemented
        # Unlike the supplied solution, NEVER reduce integer exponents mod n.
        # Built-in three-argument pow is logarithmic in the exponent size.
        # Negative powers compute modular inverses when they exist.
        return Mod(pow(self.value, exp, self.modulus), self.modulus)

    def __rpow__(self, base: object) -> Mod:
        operand = self._coerce(base)
        if operand is NotImplemented:
            return NotImplemented
        return Mod(pow(operand, self.value, self.modulus), self.modulus)

    def __ipow__(self, exponent: object) -> Mod:
        result = self ** exponent
        if result is NotImplemented:
            return NotImplemented
        self._value = result.value  # only mutate after successful computation
        return self


## 3. Usage and invariants

`+=` mutates the existing object; ordinary `+` returns a new one. Python's `%` is used to normalize all stored values, including explicit `value` assignments. Reflected subtraction must reverse the operands. Negative powers exist only when the base is invertible modulo the modulus.

In [2]:
a = Mod(10, 7)
b = Mod(-2, 7)
print("normalized:", a, b)
print("addition:", a + b, "reflected subtraction:", 3 - a)
print("modular inverse:", Mod(3, 11) ** -1)
print("correct exponent:", Mod(2, 5) ** 5)
copy_reference = a
a += 5
print("in-place identity preserved:", copy_reference is a, a)


normalized: Mod(3, 7) Mod(5, 7)
addition: Mod(1, 7) reflected subtraction: Mod(0, 7)
modular inverse: Mod(4, 11)
correct exponent: Mod(2, 5)
in-place identity preserved: True Mod(1, 7)


## 4. Testing

All tests are in the notebook and run using `unittest` (no additional packages). The final exhaustive test checks **1,210** small operand/modulus combinations for each of three arithmetic operators. Tests explicitly cover reflected dispatch, mutation, failure atomicity, incompatible moduli, and modular exponentiation.

In [3]:
import unittest


class TestModAdvanced(unittest.TestCase):
    def test_canonical_values_and_repr(self):
        self.assertEqual((Mod(-1, 7).value, repr(Mod(15, 7))), (6, "Mod(1, 7)"))

    def test_modulus_one(self):
        self.assertEqual(Mod(99, 1).value, 0)

    def test_invalid_modulus(self):
        for modulus in (0, -2):
            with self.subTest(modulus=modulus), self.assertRaises(ValueError):
                Mod(1, modulus)

    def test_invalid_types_and_bool(self):
        for value, modulus in ((1.2, 7), (1, 7.0), (True, 7), (1, False)):
            with self.subTest(value=value, modulus=modulus), self.assertRaises(TypeError):
                Mod(value, modulus)

    def test_setter_normalizes(self):
        item = Mod(1, 5)
        item.value = -7
        self.assertEqual(item.value, 3)

    def test_bad_setter_atomic(self):
        item = Mod(1, 5)
        with self.assertRaises(TypeError):
            item.value = 1.5
        self.assertEqual(item.value, 1)

    def test_modulus_read_only(self):
        with self.assertRaises(AttributeError):
            Mod(1, 5).modulus = 7

    def test_congruent_integer_equality(self):
        self.assertTrue(Mod(1, 5) == 11)
        self.assertTrue(11 == Mod(1, 5))
        self.assertFalse(Mod(1, 5) == 12)

    def test_other_modulus_is_unequal(self):
        self.assertNotEqual(Mod(1, 5), Mod(1, 7))

    def test_unhashable(self):
        with self.assertRaises(TypeError):
            hash(Mod(1, 5))

    def test_ordering_by_residue(self):
        self.assertTrue(Mod(1, 5) < Mod(2, 5))
        self.assertTrue(Mod(4, 5) > 7)  # 7 -> 2
        self.assertTrue(Mod(2, 5) <= 7)

    def test_mixed_modulus_order_raises(self):
        with self.assertRaises(TypeError):
            Mod(1, 5) < Mod(2, 7)

    def test_addition(self):
        self.assertEqual((Mod(4, 7) + Mod(6, 7)).value, 3)

    def test_integer_and_reflected_addition(self):
        self.assertEqual((4 + Mod(6, 7)).value, 3)
        self.assertEqual((Mod(6, 7) + 4).value, 3)

    def test_subtraction_and_reflection(self):
        self.assertEqual((Mod(2, 7) - 6).value, 3)
        self.assertEqual((6 - Mod(2, 7)).value, 4)

    def test_multiplication_and_reflection(self):
        self.assertEqual((Mod(3, 7) * 5).value, 1)
        self.assertEqual((5 * Mod(3, 7)).value, 1)

    def test_unary_and_int(self):
        self.assertEqual(int(-Mod(3, 7)), 4)
        self.assertEqual(int(+Mod(3, 7)), 3)

    def test_inplace_identity_and_alias(self):
        item = Mod(4, 7)
        alias = item
        item += 5
        self.assertIs(item, alias)
        self.assertEqual(alias.value, 2)
        item -= 4
        item *= 3
        self.assertEqual(item.value, 1)

    def test_incompatible_operands(self):
        with self.assertRaises(TypeError):
            Mod(1, 5) + Mod(1, 7)
        with self.assertRaises(TypeError):
            Mod(1, 5) * 3.5

    def test_power_does_not_reduce_exponent(self):
        self.assertEqual((Mod(2, 5) ** 5).value, 2)  # 2**5 % 5 != 2**0 % 5

    def test_power_large_exponent(self):
        self.assertEqual((Mod(7, 101) ** 10**6).value, pow(7, 10**6, 101))

    def test_power_mod_exponent_and_reflected(self):
        self.assertEqual((Mod(2, 7) ** Mod(3, 7)).value, 1)
        self.assertEqual((2 ** Mod(3, 7)).value, 1)

    def test_negative_power(self):
        self.assertEqual((Mod(3, 11) ** -1).value, 4)
        self.assertEqual((Mod(3, 11) ** -2).value, 5)

    def test_negative_power_nonunit(self):
        with self.assertRaises(ValueError):
            Mod(2, 4) ** -1

    def test_power_rejects_third_modulus(self):
        with self.assertRaises(TypeError):
            pow(Mod(2, 7), 3, 7)

    def test_inplace_power_failure_is_atomic(self):
        item = Mod(2, 4)
        with self.assertRaises(ValueError):
            item **= -1
        self.assertEqual(item.value, 2)

    def test_inplace_power_correctness(self):
        item = Mod(2, 5)
        item **= 5
        self.assertEqual(item.value, 2)

    def test_inplace_incompatible_is_atomic(self):
        item = Mod(2, 5)
        with self.assertRaises(TypeError):
            item += Mod(2, 7)
        self.assertEqual(item.value, 2)

    def test_exhaustive_small_arithmetic(self):
        for modulus in range(1, 11):
            for a in range(-5, 6):
                for b in range(-5, 6):
                    x, y = Mod(a, modulus), Mod(b, modulus)
                    self.assertEqual((x+y).value, (a+b) % modulus)
                    self.assertEqual((x-y).value, (a-b) % modulus)
                    self.assertEqual((x*y).value, (a*b) % modulus)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestModAdvanced)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Advanced solution tests failed"


test_addition (__main__.TestModAdvanced.test_addition) ... ok
test_bad_setter_atomic (__main__.TestModAdvanced.test_bad_setter_atomic) ... ok
test_canonical_values_and_repr (__main__.TestModAdvanced.test_canonical_values_and_repr) ... ok
test_congruent_integer_equality (__main__.TestModAdvanced.test_congruent_integer_equality) ... ok
test_exhaustive_small_arithmetic (__main__.TestModAdvanced.test_exhaustive_small_arithmetic) ... ok
test_incompatible_operands (__main__.TestModAdvanced.test_incompatible_operands) ... ok
test_inplace_identity_and_alias (__main__.TestModAdvanced.test_inplace_identity_and_alias) ... ok
test_inplace_incompatible_is_atomic (__main__.TestModAdvanced.test_inplace_incompatible_is_atomic) ... ok
test_inplace_power_correctness (__main__.TestModAdvanced.test_inplace_power_correctness) ... ok
test_inplace_power_failure_is_atomic (__main__.TestModAdvanced.test_inplace_power_failure_is_atomic) ... ok
test_integer_and_reflected_addition (__main__.TestModAdvanced.test_i

## 5. Design notes / boundaries

- Do not implement `__hash__` on this mutable, integer-congruent class. `Mod(1, 5)` is equal to both the integers `1` and `6`, which have different hashes: no single hash could satisfy the hash contract.
- Negative powers use Python's modular inverse feature (`pow(a, -1, n)`). A non-invertible element raises `ValueError` and in-place exponentiation does not mutate the object.
- Algebraic modular arithmetic does **not** define a natural ordering. The exercise's `<` and `>` are provided as a documented convenience: compare canonical residues only.
- `__slots__` and read-only properties are API controls, not security barriers. Python does not offer tamper-proof private fields.
- For hash-safe value objects, explicit congruence helpers, modular division, CRT, and linear-congruence solving, open the **separate, standalone extended notebook**.